In [ ]:
from ipydatagrid import DataGrid
from json import load
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from io import StringIO
from datetime import datetime
from collections import namedtuple
import os
from dateutil.relativedelta import relativedelta
import calendar
import matplotlib.dates as mdates

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
lmap = lambda func, l: list(map(func, l))

In [ ]:
def remove_readme(file_list: list[str]) -> list[str]:
    return [f for f in file_list if f!="readme.md"]

def get_budget_files() -> list[str]:
    return remove_readme(os.listdir("./budgets"))

def get_income_files() -> list[str]:
    return remove_readme(os.listdir("./income"))

if not get_budget_files():
    raise RuntimeError("No budget csv")
if not get_income_files():
    raise RuntimeError("No income csv")

In [ ]:
Budget = tuple[datetime, pd.DataFrame]

def parse_budget_date(filename: str) -> datetime:
    return datetime.strptime(filename[:10], "%Y-%m-%d")

def get_budget(filename: str) -> Budget:
    return (parse_budget_date(filename), pd.read_csv(f"./budgets/{filename}", index_col="item"))

def get_income(filename: str) -> Budget:
    return (parse_budget_date(filename), pd.read_csv(f"./income/{filename}", index_col="item"))

In [ ]:
budgets: list[Budget] = lmap(get_budget, get_budget_files())
budgets

In [ ]:
Income = tuple[datetime, pd.DataFrame]

income: list[Income] = lmap(get_income, get_income_files())
income[0][1]

In [ ]:
current_budget_date, current_budget = budgets[-1]
current_budget

In [ ]:
def fractional_months_between(last_recording_date: datetime, current_recording_date: datetime) -> float:
    delta = relativedelta(current_recording_date, last_recording_date)
    months = delta.months
    days = delta.days
    _, days_in_month = calendar.monthrange(current_recording_date.year, current_recording_date.month)

    return months + (days / days_in_month)

fractional_months_between(datetime(2025, 2, 1), datetime(2025, 2, 2))

In [ ]:
def budgeted_spending_between_dates(start_date: datetime, end_date: datetime, budgets: list[Budget]):
    months = fractional_months_between(start_date, end_date)
    return budgets[0][1]["cost"] * months

def income_between_dates(start_date: datetime, end_date: datetime, income: list[Income]):
    months = (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month)
    return income[0][1]["income"] * months

budgeted_spending_between_dates(datetime(2025, 2, 1), datetime(2025, 6, 2), budgets)

In [ ]:
balence = pd.read_csv("./balence_record.csv")
balence["Date"] = pd.to_datetime(balence["Date"]).dt.date

balence['Total'] = balence.drop(columns=['Date']).sum(axis=1)
balence

In [ ]:
balence["previous total"] = balence["Total"].shift(1)
dates = balence["Date"]

date_convolution = list(zip(dates[1:], dates))
balence["months since last recording"] = [np.NaN] + [fractional_months_between(last_date, date) for date, last_date in date_convolution]

balence["expected income since last recording"] = [np.NaN] + [sum(income_between_dates(last_date, date, income)) for date, last_date in date_convolution]

balence["predicted spend since last recording"] = [np.NaN] + [sum(budgeted_spending_between_dates(last_date, date, budgets)) for date, last_date in date_convolution]
balence["actual spend since last recording"] = - balence["Total"] + balence["previous total"] + balence["expected income since last recording"]
balence["spending deficit"] = balence["predicted spend since last recording"] - balence["actual spend since last recording"]
balence["predicted spending rate since last recording (£/m)"] = balence["predicted spend since last recording"] / balence["months since last recording"]
balence["actual spending rate since last recording (£/m)"] = balence["actual spend since last recording"] / balence["months since last recording"]


balence["predicted total from last recording"] = balence["previous total"] + balence["expected income since last recording"] - balence["predicted spend since last recording"]
balence["predicted total deficit"] = balence["Total"] - balence["predicted total from last recording"]

# saving since last recording
balence["balance change"] = balence["Total"] - balence["previous total"]
balence["saving rate since last recording"] = balence["balance change"] / balence["months since last recording"]
balence["predicted saving rate"] = (balence["expected income since last recording"] - balence["predicted spend since last recording"]) / balence["months since last recording"]

# predicted latest total
latest_date = np.max(balence["Date"])
latest_total = balence.loc[(balence["Date"] == latest_date).idxmax()]["Total"]
balence["predicted latest total"] = [total + sum(income_between_dates(date, latest_date, income)) - sum(budgeted_spending_between_dates(date, latest_date, budgets)) 
                                     for _, (date, total) in balence[["Date", "Total"]].iterrows()]
balence["latest total difference"] = latest_total - balence["predicted latest total"]

balence


In [ ]:
df = current_budget
item_sizes = df['cost']
item_labels = df.index
item_colors = plt.get_cmap('tab20')(range(len(df)))

category_grouped = df.groupby('category', sort=False)['cost'].sum()
category_labels = category_grouped.index
category_sizes = category_grouped.values
category_colors = plt.get_cmap('tab10')(range(len(category_grouped)))

category_color_map = dict(zip(category_labels, category_colors))
outer_colors = [category_color_map[cat] for cat in df['category']]

fig, ax = plt.subplots(figsize=(10, 8))

wedges_inner, _ = ax.pie(
    item_sizes,
    radius=0.7,
    labels=None,
    colors=item_colors,
    wedgeprops=dict(width=0.3, edgecolor='white')
)

wedges_outer, _ = ax.pie(
    category_sizes,
    radius=1,
    labels=category_labels,
    colors=category_colors,
    labeldistance=1.05,
    wedgeprops=dict(width=0.3, edgecolor='white')
)

ax.legend(wedges_inner, item_labels, title="Items", bbox_to_anchor=(1, 1), loc="upper left")

ax.set(aspect="equal", title='Monthly Expenditure by Item and Category')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Total Balance and Predicted Balance Over Time
plt.figure()
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.plot(balence["Date"], balence["Total"], label="Actual Total")
plt.plot(balence["Date"], balence["predicted total from last recording"], label="Predicted Total")
plt.title("Actual vs Predicted Balance Over Time")
plt.xlabel("Date")
plt.ylabel("Total Amount")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 3. Predicted vs Actual Savings Rate
plt.figure()
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.plot(balence["Date"], balence["saving rate since last recording"], label="Saving Rate")
plt.plot(balence["Date"], balence["predicted saving rate"], label="predicted saving rate")
plt.title("Saving Rate Since Last Recording")
plt.xlabel("Date")
plt.ylabel("Monthly Saving Rate")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
start_date = balence["Date"].min()
start_total = balence.loc[balence["Date"] == start_date, "Total"].values[0]
# Generate list of 4ths up to today or a bit into the future
end_date = balence["Date"].max() + relativedelta(months=1)
dates = []
current_date = start_date.replace(day=1)

while current_date <= end_date:
    first = current_date.replace(day=1)
    last = (current_date + relativedelta(months=1)) - pd.Timedelta(days=1)
    dates.append(first)
    dates.append(last)
    current_date += relativedelta(months=1)

# Compute predicted balance at each 4th of the month
predicted_balances = []
previous_date = start_date
current_balance = start_total

for date in dates:
    predicted_income = sum(income_between_dates(previous_date, date, income))
    spend = sum(budgeted_spending_between_dates(previous_date, date, budgets))
    current_balance += predicted_income - spend
    predicted_balances.append(current_balance)
    previous_date = date

# Build the DataFrame
predicted_balence = pd.DataFrame({
    "Date": dates,
    "Predicted Balance": predicted_balances
})

plt.figure()
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.plot(balence["Date"], balence["Total"], marker="+", label="actual balence")
plt.plot(predicted_balence["Date"], predicted_balence["Predicted Balance"], linestyle="dotted", label="predicted balence")

# Draw vertical lines connecting actual to predicted balance (when dates match)
for _, actual_row in balence.iterrows():
    date = actual_row["Date"]
    actual = actual_row["Total"]
    
    # Find matching prediction
    match = predicted_balence[predicted_balence["Date"] == date]
    if not match.empty:
        predicted = match["Predicted Balance"].values[0]
        plt.plot([date, date], [actual, predicted], color="gray", linestyle="dotted")

plt.legend()
plt.title("Balence over time")
plt.xlabel("Date")
plt.ylabel("balence (£)")
plt.grid(True)
plt.show()